# Data Preparation

## Objective

This notebook transforms the audited brain tumor MRI dataset into a
reproducible experimental dataset for image classification.

The preparation pipeline is designed to be **model-independent** so that the
same train, validation, and test partitions can be reused across different
architectures.

Based on the findings of the data audit, the main objectives are to:

- define the final 10-class pathology classification target;
- reconstruct the audited `split_group` structure;
- create leakage-aware train, validation, and test partitions at the
  `split_group` level;
- preserve pathology representation while accounting for unequal group sizes;
- verify pathology and MRI modality distributions across partitions;
- generate a reproducible image-level manifest containing the final split
  assignment;
- keep model-specific transformations separate from the shared dataset
  preparation pipeline.

The resulting partitions will remain fixed across subsequent model experiments,
allowing fair comparison between architectures.

Because explicit patient identifiers are unavailable, `split_group` represents
a reconstructed case-level proxy rather than a verified patient identifier.

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

import re
from pathlib import PureWindowsPath

import hashlib
from PIL import Image
from itertools import combinations

from collections import Counter

## 1. Load audited dataset metadata

The preparation pipeline starts from the original dataset metadata rather than
from objects stored in the previous exploratory notebook.

This ensures that the preparation process can be reproduced independently from
`01_data_audit.ipynb`.

In [3]:
PROJECT_ROOT = Path.cwd().parent

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

DATA_JSON = RAW_DATA_DIR / "DATA.json"


In [4]:
with open(DATA_JSON, "r", encoding="utf-8") as f:
    metadata = json.load(f)

type(metadata), len(metadata)

(dict, 11300)

In [8]:
df = (
    pd.DataFrame.from_dict(metadata, orient="index")
    .rename_axis("relative_path")
    .reset_index()
)

df.shape

(11300, 7)

In [9]:
df.head()

,relative_path,width,height,point,location,description,class
0,Astrocytoma T1\T1 - Anaplastic astrocytoma (pi...,512,512,"{'x': 269, 'y': 286}",[],There is a mass in the quadrigeminal plate cis...,Astrocytoma T1
1,Astrocytoma T1\T1 - Anaplastic astrocytoma (pi...,512,512,"{'x': 260, 'y': 291}",[],There is a mass in the quadrigeminal plate cis...,Astrocytoma T1
2,Astrocytoma T1\T1 - Anaplastic astrocytoma (pi...,512,512,"{'x': 259, 'y': 299}",[],There is a mass in the quadrigeminal plate cis...,Astrocytoma T1
3,Astrocytoma T1\T1 - Anaplastic astrocytoma (pi...,512,512,"{'x': 259, 'y': 305}",[],There is a mass in the quadrigeminal plate cis...,Astrocytoma T1
4,Astrocytoma T1\T1 - Anaplastic astrocytoma fro...,512,512,"{'x': 267, 'y': 158}","[frontal, ventricle]","There is a large, ill-defined region of white ...",Astrocytoma T1


In [10]:
df.columns.tolist()

['relative_path',
 'width',
 'height',
 'point',
 'location',
 'description',
 'class']

In [11]:
df["relative_path"].head(10).tolist()

['Astrocytoma T1\\T1 - Anaplastic astrocytoma (pineal region) 001.jpg',
 'Astrocytoma T1\\T1 - Anaplastic astrocytoma (pineal region) 002.jpg',
 'Astrocytoma T1\\T1 - Anaplastic astrocytoma (pineal region) 003.jpg',
 'Astrocytoma T1\\T1 - Anaplastic astrocytoma (pineal region) 004.jpg',
 'Astrocytoma T1\\T1 - Anaplastic astrocytoma frontal , ventricle 001.jpg',
 'Astrocytoma T1\\T1 - Anaplastic astrocytoma frontal , ventricle 002.jpg',
 'Astrocytoma T1\\T1 - Anaplastic astrocytoma frontal , ventricle 003.jpg',
 'Astrocytoma T1\\T1 - Anaplastic astrocytoma NOS 004.jpg',
 'Astrocytoma T1\\T1 - Anaplastic astrocytoma NOS 005.jpg',
 'Astrocytoma T1\\T1 - Anaplastic astrocytoma NOS 006.jpg']

In [13]:

def parse_class_label(class_label):
    if class_label.endswith(" T1C+"):
        return class_label[:-5], "T1C+"
    elif class_label.endswith(" T1"):
        return class_label[:-3], "T1"
    elif class_label.endswith(" T2"):
        return class_label[:-3], "T2"
    else:
        raise ValueError(f"Unexpected class label: {class_label}")

df[["pathology", "modality"]] = df["class"].apply(
    lambda x: pd.Series(parse_class_label(x))
)

df["filename"] = df["relative_path"].apply(
    lambda x: PureWindowsPath(x).name
)

df[["relative_path", "pathology", "modality", "filename"]].head(10)

,relative_path,pathology,modality,filename
0,Astrocytoma T1\T1 - Anaplastic astrocytoma (pi...,Astrocytoma,T1,T1 - Anaplastic astrocytoma (pineal region) 00...
1,Astrocytoma T1\T1 - Anaplastic astrocytoma (pi...,Astrocytoma,T1,T1 - Anaplastic astrocytoma (pineal region) 00...
2,Astrocytoma T1\T1 - Anaplastic astrocytoma (pi...,Astrocytoma,T1,T1 - Anaplastic astrocytoma (pineal region) 00...
3,Astrocytoma T1\T1 - Anaplastic astrocytoma (pi...,Astrocytoma,T1,T1 - Anaplastic astrocytoma (pineal region) 00...
4,Astrocytoma T1\T1 - Anaplastic astrocytoma fro...,Astrocytoma,T1,"T1 - Anaplastic astrocytoma frontal , ventricl..."
5,Astrocytoma T1\T1 - Anaplastic astrocytoma fro...,Astrocytoma,T1,"T1 - Anaplastic astrocytoma frontal , ventricl..."
6,Astrocytoma T1\T1 - Anaplastic astrocytoma fro...,Astrocytoma,T1,"T1 - Anaplastic astrocytoma frontal , ventricl..."
7,Astrocytoma T1\T1 - Anaplastic astrocytoma NOS...,Astrocytoma,T1,T1 - Anaplastic astrocytoma NOS 004.jpg
8,Astrocytoma T1\T1 - Anaplastic astrocytoma NOS...,Astrocytoma,T1,T1 - Anaplastic astrocytoma NOS 005.jpg
9,Astrocytoma T1\T1 - Anaplastic astrocytoma NOS...,Astrocytoma,T1,T1 - Anaplastic astrocytoma NOS 006.jpg


In [14]:
print("Pathologies:", sorted(df["pathology"].unique()))
print("Modalities:", sorted(df["modality"].unique()))
print("Number of pathologies:", df["pathology"].nunique())
print("Number of modalities:", df["modality"].nunique())

Pathologies: ['Astrocytoma', 'Ependymoma', 'Glioma', 'Hemangiopericytoma', 'Meningioma', 'Neurocytoma', 'Normal', 'Oligodendroglioma', 'Other', 'Schwannoma']
Modalities: ['T1', 'T1C+', 'T2']
Number of pathologies: 10
Number of modalities: 3


In [ ]:
def extract_case_name(filename):
    name = filename.rsplit(".", 1)[0]

    # Remove modality prefix
    name = re.sub(r"^(T1C\+|T1|T2)\s*-\s*", "", name)

    # Remove trailing image number
    name = re.sub(r"\s+\d{3}$", "", name)

    return name.strip()

df["case_name"] = df["filename"].apply(extract_case_name)

df[["filename", "case_name"]].head(15)

,filename,case_name
0,T1 - Anaplastic astrocytoma (pineal region) 00...,Anaplastic astrocytoma (pineal region)
1,T1 - Anaplastic astrocytoma (pineal region) 00...,Anaplastic astrocytoma (pineal region)
2,T1 - Anaplastic astrocytoma (pineal region) 00...,Anaplastic astrocytoma (pineal region)
3,T1 - Anaplastic astrocytoma (pineal region) 00...,Anaplastic astrocytoma (pineal region)
4,"T1 - Anaplastic astrocytoma frontal , ventricl...","Anaplastic astrocytoma frontal , ventricle"
5,"T1 - Anaplastic astrocytoma frontal , ventricl...","Anaplastic astrocytoma frontal , ventricle"
6,"T1 - Anaplastic astrocytoma frontal , ventricl...","Anaplastic astrocytoma frontal , ventricle"
7,T1 - Anaplastic astrocytoma NOS 004.jpg,Anaplastic astrocytoma NOS
8,T1 - Anaplastic astrocytoma NOS 005.jpg,Anaplastic astrocytoma NOS
9,T1 - Anaplastic astrocytoma NOS 006.jpg,Anaplastic astrocytoma NOS


In [16]:
df["case_name"].nunique()

274

In [17]:
df.groupby(["pathology", "case_name"]).size().sort_values(ascending=False).head(20)

pathology           case_name                                                                                  
Normal              MRI brain - normal (70-year-old) ventricle , brainstem                                         310
Oligodendroglioma   Oligodendroglioma frontal                                                                      291
Neurocytoma         Central neurocytoma ventricle                                                                  245
Meningioma          Meningioma frontal                                                                             227
Schwannoma          Trigeminal schwannoma trigeminal , cerebellopontine angle                                      226
Meningioma          Meningioma                                                                                     218
Glioma              Bilateral thalamic glioma ventricle                                                            215
Hemangiopericytoma  Solitary fibrous tumor (hemangioper

In [18]:
case_groups = (
    df[["pathology", "case_name"]]
    .drop_duplicates()
    .sort_values(["pathology", "case_name"])
    .reset_index(drop=True)
)

case_groups["case_group_id"] = np.arange(len(case_groups))

df = df.merge(
    case_groups,
    on=["pathology", "case_name"],
    how="left",
    validate="many_to_one"
)

df["case_group_id"].nunique()

274

In [19]:
def extract_image_number(filename):
    match = re.search(r"(\d{3})\.jpg$", filename)
    return int(match.group(1)) if match else np.nan

df["image_number"] = df["filename"].apply(extract_image_number)

df[
    ["filename", "case_name", "pathology", "modality", "image_number"]
].head(15)

,filename,case_name,pathology,modality,image_number
0,T1 - Anaplastic astrocytoma (pineal region) 00...,Anaplastic astrocytoma (pineal region),Astrocytoma,T1,1
1,T1 - Anaplastic astrocytoma (pineal region) 00...,Anaplastic astrocytoma (pineal region),Astrocytoma,T1,2
2,T1 - Anaplastic astrocytoma (pineal region) 00...,Anaplastic astrocytoma (pineal region),Astrocytoma,T1,3
3,T1 - Anaplastic astrocytoma (pineal region) 00...,Anaplastic astrocytoma (pineal region),Astrocytoma,T1,4
4,"T1 - Anaplastic astrocytoma frontal , ventricl...","Anaplastic astrocytoma frontal , ventricle",Astrocytoma,T1,1
5,"T1 - Anaplastic astrocytoma frontal , ventricl...","Anaplastic astrocytoma frontal , ventricle",Astrocytoma,T1,2
6,"T1 - Anaplastic astrocytoma frontal , ventricl...","Anaplastic astrocytoma frontal , ventricle",Astrocytoma,T1,3
7,T1 - Anaplastic astrocytoma NOS 004.jpg,Anaplastic astrocytoma NOS,Astrocytoma,T1,4
8,T1 - Anaplastic astrocytoma NOS 005.jpg,Anaplastic astrocytoma NOS,Astrocytoma,T1,5
9,T1 - Anaplastic astrocytoma NOS 006.jpg,Anaplastic astrocytoma NOS,Astrocytoma,T1,6


In [20]:
df["image_number"].isna().sum()

np.int64(0)

In [21]:
sequence_check = (
    df.groupby(["pathology", "case_name", "modality"])["image_number"]
      .agg(
          n_images="count",
          min_number="min",
          max_number="max",
          n_unique="nunique"
      )
      .reset_index()
)

sequence_check.head(30)

,pathology,case_name,modality,n_images,min_number,max_number,n_unique
0,Astrocytoma,Anaplastic astrocytoma (pineal region),T1,4,1,4,4
1,Astrocytoma,Anaplastic astrocytoma (pineal region),T1C+,3,1,3,3
2,Astrocytoma,Anaplastic astrocytoma (pineal region),T2,4,1,4,4
3,Astrocytoma,Anaplastic astrocytoma NOS,T1,3,4,6,3
4,Astrocytoma,Anaplastic astrocytoma NOS,T2,3,4,6,3
5,Astrocytoma,"Anaplastic astrocytoma frontal , ventricle",T1,3,1,3,3
6,Astrocytoma,"Anaplastic astrocytoma frontal , ventricle",T1C+,6,1,6,6
7,Astrocytoma,"Anaplastic astrocytoma frontal , ventricle",T2,5,1,5,5
8,Astrocytoma,"Anaplastic astrocytoma parietal , ventricle",T1,25,1,25,25
9,Astrocytoma,"Anaplastic astrocytoma parietal , ventricle",T1C+,23,1,23,23


In [22]:
sequence_check[
    sequence_check["n_images"] != sequence_check["max_number"] - sequence_check["min_number"] + 1
].head(30)

,pathology,case_name,modality,n_images,min_number,max_number,n_unique
83,Astrocytoma,Pilocytic astrocytoma ventricle,T2,11,19,46,11
258,Hemangiopericytoma,Meningeal hemangiopericytoma (pediatric) front...,T1,7,6,13,7
274,Hemangiopericytoma,Solitary fibrous tumor (hemangiopericytoma) wi...,T1,77,29,106,77


In [23]:
description_clean = df["description"].astype("string").str.strip()

has_description = (
    description_clean.notna()
    & description_clean.ne("")
)

df["case_key"] = np.where(
    has_description,
    "DESC::" + description_clean,
    "NODESC::" + df["pathology"] + "::" + df["case_name"]
)

df["case_group_id"], _ = pd.factorize(
    df["case_key"],
    sort=False
)

print("Initial case groups:", df["case_group_id"].nunique())

Initial case groups: 361


In [24]:
no_description_groups = (
    df.loc[~has_description]
      .groupby(["case_group_id", "pathology", "case_name"])
      .size()
      .reset_index(name="n_images")
)

no_description_groups

,case_group_id,pathology,case_name,n_images
0,226,Normal,Normal brain MRI (non-focal epilepsy protocol),105
1,278,Other,Choroid plexus carcinoma,149
2,334,Schwannoma,Trigeminal schwannoma,37


In [25]:
AUDITED_GROUP_IDS = [92, 98, 131, 286, 133, 291, 228, 234]

(
    df[df["case_group_id"].isin(AUDITED_GROUP_IDS)]
    .groupby("case_group_id")
    .agg(
        pathology=("pathology", "first"),
        n_images=("relative_path", "size"),
        modalities=("modality", lambda x: sorted(x.unique())),
        case_names=("case_name", lambda x: sorted(x.unique())),
        n_descriptions=("description", "nunique")
    )
)

,pathology,n_images,modalities,case_names,n_descriptions
case_group_id,,,,,
92,Glioma,39,"[T1, T1C+, T2]",[Brainstem glioma ventricle],1
98,Glioma,14,"[T1, T1C+, T2]",[Diffuse glioma],1
131,Glioma,15,[T2],"[Low grade glioma (MR spectroscopy) temporal ,...",1
133,Hemangiopericytoma,15,"[T1, T1C+]",[Meningeal hemangiopericytoma (pediatric) fron...,1
228,Normal,163,"[T1, T1C+, T2]",[Normal brain MRI with post-contrast FLAIR],1
234,Normal,32,[T2],[Normal MRI brain - neurodegenerative protocol...,1
286,Other,25,"[T1, T1C+, T2]",[Colloid cyst supratentorial],1
291,Other,36,"[T1, T2]",[Compressive arachnoid cyst middle cranial fossa],1


## 2. Reconstruct leakage-aware case groups

The initial metadata reconstruction produces 361 case-level groups.

During the data audit, exact pixel-level duplicate analysis revealed a small
number of relationships between otherwise distinct metadata groups.

Rather than relying on notebook-specific numerical group identifiers, these
relationships are reconstructed directly from the image content.

Groups connected by exact duplicate images will be merged into a common
`split_group`, ensuring that exact duplicates cannot later be assigned to
different dataset partitions.

In [ ]:


def build_image_path(relative_path):
    return RAW_DATA_DIR.joinpath(*PureWindowsPath(relative_path).parts)

df["image_path"] = df["relative_path"].apply(build_image_path)

df[["relative_path", "image_path"]].head()

,relative_path,image_path
0,Astrocytoma T1\T1 - Anaplastic astrocytoma (pi...,/home/swsq2526/brain-tumor-mri/data/raw/Astroc...
1,Astrocytoma T1\T1 - Anaplastic astrocytoma (pi...,/home/swsq2526/brain-tumor-mri/data/raw/Astroc...
2,Astrocytoma T1\T1 - Anaplastic astrocytoma (pi...,/home/swsq2526/brain-tumor-mri/data/raw/Astroc...
3,Astrocytoma T1\T1 - Anaplastic astrocytoma (pi...,/home/swsq2526/brain-tumor-mri/data/raw/Astroc...
4,Astrocytoma T1\T1 - Anaplastic astrocytoma fro...,/home/swsq2526/brain-tumor-mri/data/raw/Astroc...


In [27]:
df["image_path"].apply(lambda p: p.exists()).value_counts()

image_path
True     11278
False       22
Name: count, dtype: int64

In [28]:
missing_mask = ~df["image_path"].apply(lambda p: p.exists())

missing_paths = df.loc[
    missing_mask,
    ["relative_path", "class", "filename"]
].copy()

print("Missing files:", len(missing_paths))

missing_paths

Missing files: 22


,relative_path,class,filename
2985,"Glioma T1C+\T1C+ - Diffuse midline glioma, H3 ...",Glioma T1C+,"T1C+ - Diffuse midline glioma, H3 K27M–mutant ..."
2986,"Glioma T1C+\T1C+ - Diffuse midline glioma, H3 ...",Glioma T1C+,"T1C+ - Diffuse midline glioma, H3 K27M–mutant ..."
2987,"Glioma T1C+\T1C+ - Diffuse midline glioma, H3 ...",Glioma T1C+,"T1C+ - Diffuse midline glioma, H3 K27M–mutant ..."
2988,"Glioma T1C+\T1C+ - Diffuse midline glioma, H3 ...",Glioma T1C+,"T1C+ - Diffuse midline glioma, H3 K27M–mutant ..."
2989,"Glioma T1C+\T1C+ - Diffuse midline glioma, H3 ...",Glioma T1C+,"T1C+ - Diffuse midline glioma, H3 K27M–mutant ..."
2990,"Glioma T1C+\T1C+ - Diffuse midline glioma, H3 ...",Glioma T1C+,"T1C+ - Diffuse midline glioma, H3 K27M–mutant ..."
2991,"Glioma T1C+\T1C+ - Diffuse midline glioma, H3 ...",Glioma T1C+,"T1C+ - Diffuse midline glioma, H3 K27M–mutant ..."
2992,"Glioma T1C+\T1C+ - Diffuse midline glioma, H3 ...",Glioma T1C+,"T1C+ - Diffuse midline glioma, H3 K27M–mutant ..."
2993,"Glioma T1C+\T1C+ - Diffuse midline glioma, H3 ...",Glioma T1C+,"T1C+ - Diffuse midline glioma, H3 K27M–mutant ..."
3464,"Glioma T2\T2 - Diffuse midline glioma, H3 K27M...",Glioma T2,"T2 - Diffuse midline glioma, H3 K27M–mutant ve..."


In [29]:
actual_images = list(RAW_DATA_DIR.rglob("*.jpg"))

actual_by_filename = {}

for path in actual_images:
    actual_by_filename.setdefault(path.name, []).append(path)

missing_paths["matches_on_disk"] = missing_paths["filename"].apply(
    lambda name: [
        str(p.relative_to(RAW_DATA_DIR))
        for p in actual_by_filename.get(name, [])
    ]
)

missing_paths[
    ["relative_path", "matches_on_disk"]
]

,relative_path,matches_on_disk
2985,"Glioma T1C+\T1C+ - Diffuse midline glioma, H3 ...",[]
2986,"Glioma T1C+\T1C+ - Diffuse midline glioma, H3 ...",[]
2987,"Glioma T1C+\T1C+ - Diffuse midline glioma, H3 ...",[]
2988,"Glioma T1C+\T1C+ - Diffuse midline glioma, H3 ...",[]
2989,"Glioma T1C+\T1C+ - Diffuse midline glioma, H3 ...",[]
2990,"Glioma T1C+\T1C+ - Diffuse midline glioma, H3 ...",[]
2991,"Glioma T1C+\T1C+ - Diffuse midline glioma, H3 ...",[]
2992,"Glioma T1C+\T1C+ - Diffuse midline glioma, H3 ...",[]
2993,"Glioma T1C+\T1C+ - Diffuse midline glioma, H3 ...",[]
3464,"Glioma T2\T2 - Diffuse midline glioma, H3 K27M...",[]


In [30]:
candidates = [
    p for p in actual_images
    if "Diffuse midline glioma" in p.name
]

print("Candidates found:", len(candidates))

for p in candidates:
    print(p.relative_to(RAW_DATA_DIR))

Candidates found: 192
Glioma T1/T1 - Diffuse midline glioma ventricle 010.jpg
Glioma T1/T1 - Diffuse midline glioma 015.jpg
Glioma T1/T1 - Diffuse midline glioma 017.jpg
Glioma T1/T1 - Diffuse midline glioma 013.jpg
Glioma T1/T1 - Diffuse midline glioma 011.jpg
Glioma T1/T1 - Diffuse midline glioma NOS ventricle 009.jpg
Glioma T1/T1 - Diffuse midline glioma NOS ventricle 010.jpg
Glioma T1/T1 - Diffuse midline glioma brainstem , ventricle 005.jpg
Glioma T1/T1 - Diffuse midline glioma brainstem , ventricle 004.jpg
Glioma T1/T1 - Diffuse midline glioma NOS ventricle 015.jpg
Glioma T1/T1 - Diffuse midline glioma ventricle 012.jpg
Glioma T1/T1 - Diffuse midline glioma 014.jpg
Glioma T1/T1 - Diffuse midline glioma 010.jpg
Glioma T1/T1 - Diffuse midline glioma brainstem , ventricle 002.jpg
Glioma T1/T1 - Diffuse midline glioma NOS ventricle 006.jpg
Glioma T1/T1 - Diffuse midline glioma NOS ventricle 013.jpg
Glioma T1/T1 - Diffuse midline glioma 019.jpg
Glioma T1/T1 - Diffuse midline glioma 01

In [31]:
for p in candidates[:10]:
    print(repr(p.name))

'T1 - Diffuse midline glioma ventricle 010.jpg'
'T1 - Diffuse midline glioma 015.jpg'
'T1 - Diffuse midline glioma 017.jpg'
'T1 - Diffuse midline glioma 013.jpg'
'T1 - Diffuse midline glioma 011.jpg'
'T1 - Diffuse midline glioma NOS ventricle 009.jpg'
'T1 - Diffuse midline glioma NOS ventricle 010.jpg'
'T1 - Diffuse midline glioma brainstem , ventricle 005.jpg'
'T1 - Diffuse midline glioma brainstem , ventricle 004.jpg'
'T1 - Diffuse midline glioma NOS ventricle 015.jpg'


In [32]:
for name in missing_paths["filename"].head(10):
    print(repr(name))

'T1C+ - Diffuse midline glioma, H3 K27M–mutant ventricle , frontal 001.jpg'
'T1C+ - Diffuse midline glioma, H3 K27M–mutant ventricle , frontal 002.jpg'
'T1C+ - Diffuse midline glioma, H3 K27M–mutant ventricle , frontal 003.jpg'
'T1C+ - Diffuse midline glioma, H3 K27M–mutant ventricle , frontal 004.jpg'
'T1C+ - Diffuse midline glioma, H3 K27M–mutant ventricle , frontal 005.jpg'
'T1C+ - Diffuse midline glioma, H3 K27M–mutant ventricle , frontal 006.jpg'
'T1C+ - Diffuse midline glioma, H3 K27M–mutant ventricle , frontal 007.jpg'
'T1C+ - Diffuse midline glioma, H3 K27M–mutant ventricle , frontal 008.jpg'
'T1C+ - Diffuse midline glioma, H3 K27M–mutant ventricle , frontal 009.jpg'
'T2 - Diffuse midline glioma, H3 K27M–mutant ventricle , frontal 001.jpg'


In [33]:
import unicodedata

def normalize_filename(name):
    name = unicodedata.normalize("NFKC", name)
    name = name.replace("–", "-").replace("—", "-")
    return name.casefold().strip()

actual_by_normalized_name = {}

for path in actual_images:
    key = normalize_filename(path.name)
    actual_by_normalized_name.setdefault(key, []).append(path)

missing_paths["normalized_matches"] = missing_paths["filename"].apply(
    lambda name: [
        str(p.relative_to(RAW_DATA_DIR))
        for p in actual_by_normalized_name.get(
            normalize_filename(name), []
        )
    ]
)

missing_paths[
    ["relative_path", "normalized_matches"]
]

,relative_path,normalized_matches
2985,"Glioma T1C+\T1C+ - Diffuse midline glioma, H3 ...",[]
2986,"Glioma T1C+\T1C+ - Diffuse midline glioma, H3 ...",[]
2987,"Glioma T1C+\T1C+ - Diffuse midline glioma, H3 ...",[]
2988,"Glioma T1C+\T1C+ - Diffuse midline glioma, H3 ...",[]
2989,"Glioma T1C+\T1C+ - Diffuse midline glioma, H3 ...",[]
2990,"Glioma T1C+\T1C+ - Diffuse midline glioma, H3 ...",[]
2991,"Glioma T1C+\T1C+ - Diffuse midline glioma, H3 ...",[]
2992,"Glioma T1C+\T1C+ - Diffuse midline glioma, H3 ...",[]
2993,"Glioma T1C+\T1C+ - Diffuse midline glioma, H3 ...",[]
3464,"Glioma T2\T2 - Diffuse midline glioma, H3 K27M...",[]


In [34]:
import unicodedata

def normalize_relative_path(path):
    path = str(path)

    # Standardize path separators
    path = path.replace("\\", "/")

    # Normalize Unicode representation
    path = unicodedata.normalize("NFKC", path)

    # Fix known mojibake from the dataset filenames
    path = path.replace("â€“", "–")

    return path

In [35]:
actual_images = list(RAW_DATA_DIR.rglob("*.jpg"))

disk_path_lookup = {
    normalize_relative_path(path.relative_to(RAW_DATA_DIR).as_posix()): path
    for path in actual_images
}

print("Images found on disk:", len(actual_images))
print("Normalized paths:", len(disk_path_lookup))

Images found on disk: 11300
Normalized paths: 11300


In [36]:
df["normalized_relative_path"] = (
    df["relative_path"]
    .apply(normalize_relative_path)
)

In [37]:
df["image_path"] = (
    df["normalized_relative_path"]
    .map(disk_path_lookup)
)

In [38]:
print("Resolved:", df["image_path"].notna().sum())
print("Unresolved:", df["image_path"].isna().sum())

Resolved: 11300
Unresolved: 0


In [39]:
df["image_path"].apply(
    lambda p: p.exists() if p is not None else False
).value_counts()

image_path
True    11300
Name: count, dtype: int64

### 2.1 Exact duplicate reconstruction

All 11,300 metadata entries were successfully resolved to their corresponding
image files after normalizing path separators and a known filename encoding
issue.

Exact pixel-level hashes are now recomputed to reproduce the duplicate
relationships identified during the data audit.

In [42]:


def compute_pixel_hash(path):
    with Image.open(path) as img:
        img = img.convert("L")
        pixel_bytes = np.asarray(img).tobytes()

    return hashlib.sha256(pixel_bytes).hexdigest()

df["pixel_hash"] = df["image_path"].apply(compute_pixel_hash)

In [43]:
duplicate_mask = df.duplicated("pixel_hash", keep=False)

print("Images:", len(df))
print("Unique pixel hashes:", df["pixel_hash"].nunique())
print("Images in exact duplicate groups:", duplicate_mask.sum())
print(
    "Exact duplicate groups:",
    df.loc[duplicate_mask, "pixel_hash"].nunique()
)

Images: 11300
Unique pixel hashes: 9435
Images in exact duplicate groups: 3317
Exact duplicate groups: 1452


In [44]:
cross_group_hashes = (
    df.groupby("pixel_hash")["case_group_id"]
      .nunique()
)

cross_group_hashes = cross_group_hashes[
    cross_group_hashes > 1
].index

cross_group_duplicates = (
    df[df["pixel_hash"].isin(cross_group_hashes)]
    .sort_values(["pixel_hash", "case_group_id"])
)

print("Exact hashes crossing case groups:", len(cross_group_hashes))
print(
    "Case-group pairs involved:",
    cross_group_duplicates[
        ["pixel_hash", "case_group_id"]
    ]
    .drop_duplicates()
    .groupby("pixel_hash")["case_group_id"]
    .apply(tuple)
    .nunique()
)

Exact hashes crossing case groups: 57
Case-group pairs involved: 4


In [ ]:


group_pairs = set()

for _, group in cross_group_duplicates.groupby("pixel_hash"):
    group_ids = sorted(group["case_group_id"].unique())

    for pair in combinations(group_ids, 2):
        group_pairs.add(pair)

print("Distinct cross-group pairs:", len(group_pairs))
print(sorted(group_pairs))

Distinct cross-group pairs: 4
[(np.int64(64), np.int64(76)), (np.int64(77), np.int64(80)), (np.int64(247), np.int64(250)), (np.int64(312), np.int64(319))]


In [47]:
for group_a, group_b in sorted(group_pairs):
    subset = df[
        df["case_group_id"].isin([group_a, group_b])
    ]

    print(f"\nGroups {group_a} <-> {group_b}")
    print("Pathologies:", subset["pathology"].unique())
    print("Modalities:", sorted(subset["modality"].unique()))
    print("Case names:", subset["case_name"].unique())


Groups 64 <-> 76
Pathologies: ['Ependymoma']
Modalities: ['T1', 'T1C+', 'T2']
Case names: ['Posterior fossa ependymoma supratentorial'
 'Ependymoma intraventricular , ventricle']

Groups 77 <-> 80
Pathologies: ['Ependymoma']
Modalities: ['T1C+', 'T2']
Case names: ['Ependymoma posterior fossa , ventricle'
 'Posterior fossa ependymoma ventricle']

Groups 247 <-> 250
Pathologies: ['Oligodendroglioma']
Modalities: ['T1', 'T1C+', 'T2']
Case names: ['Oligodendroglioma frontal']

Groups 312 <-> 319
Pathologies: ['Other']
Modalities: ['T1C+', 'T2']
Case names: ['Colloid cyst ventricle' 'Colloid cyst']


In [48]:
# Start with each case group as its own component
parent = {
    group_id: group_id
    for group_id in df["case_group_id"].unique()
}

def find(x):
    if parent[x] != x:
        parent[x] = find(parent[x])
    return parent[x]

def union(a, b):
    root_a = find(a)
    root_b = find(b)

    if root_a != root_b:
        parent[root_b] = root_a


# Merge case groups connected by exact duplicate images
for group_a, group_b in group_pairs:
    union(group_a, group_b)


# Obtain the connected-component representative
df["split_group_raw"] = df["case_group_id"].apply(find)

# Convert representatives into clean sequential IDs
unique_groups = sorted(df["split_group_raw"].unique())

split_group_map = {
    group_id: new_id
    for new_id, group_id in enumerate(unique_groups)
}

df["split_group"] = df["split_group_raw"].map(split_group_map)

In [49]:
print("Initial case groups:", df["case_group_id"].nunique())
print("Final split groups:", df["split_group"].nunique())
print(
    "Reduction:",
    df["case_group_id"].nunique()
    - df["split_group"].nunique()
)

Initial case groups: 361
Final split groups: 357
Reduction: 4


In [50]:
assert df["split_group"].nunique() == 357

assert (
    df.groupby("pixel_hash")["split_group"]
      .nunique()
      .max()
    == 1
)

print("✓ 357 leakage-aware split groups reconstructed.")
print("✓ No exact pixel duplicate crosses split groups.")

✓ 357 leakage-aware split groups reconstructed.
✓ No exact pixel duplicate crosses split groups.


In [51]:
group_df = (
    df.groupby("split_group")
      .agg(
          pathology=("pathology", "first"),
          n_images=("relative_path", "size"),
          n_modalities=("modality", "nunique"),
          modalities=("modality", lambda x: tuple(sorted(x.unique())))
      )
      .reset_index()
)

group_df.head()

,split_group,pathology,n_images,n_modalities,modalities
0,0,Astrocytoma,11,3,"(T1, T1C+, T2)"
1,1,Astrocytoma,14,3,"(T1, T1C+, T2)"
2,2,Astrocytoma,6,2,"(T1, T2)"
3,3,Astrocytoma,53,3,"(T1, T1C+, T2)"
4,4,Astrocytoma,47,3,"(T1, T1C+, T2)"


In [52]:
pathologies_per_group = (
    df.groupby("split_group")["pathology"]
      .nunique()
)

assert pathologies_per_group.max() == 1

print("Groups:", len(group_df))
print("Total images:", group_df["n_images"].sum())
print("✓ Every split_group belongs to exactly one pathology.")

Groups: 357
Total images: 11300
✓ Every split_group belongs to exactly one pathology.


In [53]:
group_distribution = (
    group_df.groupby("pathology")
            .agg(
                n_groups=("split_group", "count"),
                n_images=("n_images", "sum"),
                min_images_per_group=("n_images", "min"),
                median_images_per_group=("n_images", "median"),
                max_images_per_group=("n_images", "max")
            )
            .sort_values("n_groups")
)

group_distribution

,n_groups,n_images,min_images_per_group,median_images_per_group,max_images_per_group
pathology,,,,,
Hemangiopericytoma,11,603,5,34.0,192
Normal,14,1058,15,43.5,310
Neurocytoma,15,618,11,25.0,120
Ependymoma,23,1037,11,24.0,276
Oligodendroglioma,24,550,3,16.5,100
Schwannoma,40,1236,1,19.0,226
Glioma,51,1476,5,17.0,185
Astrocytoma,57,1118,1,15.0,80
Other,57,1533,3,15.0,149


## 3. Train, validation, and test split design

The dataset will be partitioned into approximately:

- **70% training**
- **15% validation**
- **15% testing**

Splitting is performed exclusively at the `split_group` level.

A 70/15/15 allocation was selected instead of a more aggressive 80/10/10
split because several pathology classes contain relatively few independent
reconstructed groups. For example, Hemangiopericytoma, Normal, and
Neurocytoma contain only 11, 14, and 15 `split_group` units respectively.

Additionally, group sizes are highly heterogeneous. Individual groups may
contain from only a few images to several hundred images. Therefore,
stratifying solely by the number of groups could result in substantially
imbalanced image distributions across partitions.

The splitting procedure will consequently aim to satisfy two objectives:

1. preserve pathology representation across train, validation, and test;
2. approximate the desired image-level 70/15/15 proportions without ever
   dividing a `split_group`.

The resulting partitions will remain fixed across all subsequent model
experiments.

### 3.1 Initial stratified split assessment

A first 70/15/15 split was generated by stratifying reconstructed groups by
pathology.

At the global level, the resulting image distribution was reasonably close to
the desired proportions:

- Train: 71.4%
- Validation: 13.2%
- Test: 15.3%

However, substantial pathology-level imbalances remained because reconstructed
groups vary considerably in size.

For example, only 13 Hemangiopericytoma images were assigned to validation,
while 559 remained in training. Conversely, 329 of 1,037 Ependymoma images
were assigned to the test partition.

Therefore, pathology-stratified random group allocation alone is insufficient.
The final partitioning procedure should account for both pathology labels and
group image counts.

In [55]:
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

train_groups, temp_groups = train_test_split(
    group_df,
    test_size=0.30,
    stratify=group_df["pathology"],
    random_state=RANDOM_STATE
)

val_groups, test_groups = train_test_split(
    temp_groups,
    test_size=0.50,
    stratify=temp_groups["pathology"],
    random_state=RANDOM_STATE
)

print("Train groups:", len(train_groups))
print("Validation groups:", len(val_groups))
print("Test groups:", len(test_groups))

Train groups: 249
Validation groups: 54
Test groups: 54


In [56]:
split_summary = pd.DataFrame({
    "split": ["train", "validation", "test"],
    "n_groups": [
        len(train_groups),
        len(val_groups),
        len(test_groups)
    ],
    "n_images": [
        train_groups["n_images"].sum(),
        val_groups["n_images"].sum(),
        test_groups["n_images"].sum()
    ]
})

split_summary["pct_groups"] = (
    split_summary["n_groups"] / len(group_df) * 100
)

split_summary["pct_images"] = (
    split_summary["n_images"] / len(df) * 100
)

split_summary

,split,n_groups,n_images,pct_groups,pct_images
0,train,249,8069,69.747899,71.407080
1,validation,54,1497,15.126050,13.247788
2,test,54,1734,15.126050,15.345133


In [57]:
pathology_split_summary = pd.concat([
    train_groups.assign(split="train"),
    val_groups.assign(split="validation"),
    test_groups.assign(split="test")
]).pivot_table(
    index="pathology",
    columns="split",
    values=["split_group", "n_images"],
    aggfunc={
        "split_group": "count",
        "n_images": "sum"
    },
    fill_value=0
)

pathology_split_summary

n_images                  split_group                 
split                  test train validation        test train validation
pathology                                                                
Astrocytoma             170   784        164           8    40          9
Ependymoma              329   542        166           3    16          4
Glioma                  254   968        254           8    35          8
Hemangiopericytoma       31   559         13           2     8          1
Meningioma              298  1551        222          10    45         10
Neurocytoma             137   336        145           3    10          2
Normal                   73   844        141           2    10          2
Oligodendroglioma        66   393         91           4    17          3
Other                   136  1210        187           8    40          9
Schwannoma              240   882        114           6    28          6

### 3.2 Group-size-aware pathology-stratified allocation

To account for the large variation in reconstructed group sizes, the final
allocation is optimized separately within each pathology.

For each pathology:

- approximately 15% of groups are assigned to validation;
- approximately 15% of groups are assigned to test;
- at least two reconstructed groups are assigned to each evaluation partition;
- groups are selected so that the corresponding number of images is as close
  as possible to 15% of the pathology-specific image count.

This produces a deterministic group-aware allocation while avoiding the strong
image-level imbalances observed with simple random stratification.

In [58]:
def select_groups_closest_to_target(group_subset, k, target_images):
    """
    Select exactly k split_groups whose total number of images
    is as close as possible to target_images.
    """

    records = list(
        group_subset[["split_group", "n_images"]]
        .itertuples(index=False, name=None)
    )

    # dp[count][image_sum] = tuple of selected split_group IDs
    dp = [dict() for _ in range(k + 1)]
    dp[0][0] = tuple()

    for group_id, n_images in records:

        # Iterate backwards so a group cannot be reused
        for count in range(k - 1, -1, -1):

            for current_sum, selected in list(dp[count].items()):
                new_sum = current_sum + n_images

                if new_sum not in dp[count + 1]:
                    dp[count + 1][new_sum] = selected + (group_id,)

    best_sum = min(
        dp[k],
        key=lambda image_sum: abs(image_sum - target_images)
    )

    return list(dp[k][best_sum]), best_sum

In [59]:
TARGET_EVAL_RATIO = 0.15
MIN_EVAL_GROUPS = 2

split_assignments = {}

allocation_details = []

for pathology, pathology_groups in group_df.groupby("pathology"):

    pathology_groups = pathology_groups.copy()

    n_groups = len(pathology_groups)
    n_images = pathology_groups["n_images"].sum()

    # Target numbers of independent groups
    n_val_groups = max(
        MIN_EVAL_GROUPS,
        round(TARGET_EVAL_RATIO * n_groups)
    )

    n_test_groups = max(
        MIN_EVAL_GROUPS,
        round(TARGET_EVAL_RATIO * n_groups)
    )

    target_eval_images = TARGET_EVAL_RATIO * n_images

    # Validation groups
    val_ids, val_images = select_groups_closest_to_target(
        pathology_groups,
        k=n_val_groups,
        target_images=target_eval_images
    )

    remaining_groups = pathology_groups[
        ~pathology_groups["split_group"].isin(val_ids)
    ]

    # Test groups
    test_ids, test_images = select_groups_closest_to_target(
        remaining_groups,
        k=n_test_groups,
        target_images=target_eval_images
    )

    train_ids = pathology_groups[
        ~pathology_groups["split_group"].isin(
            val_ids + test_ids
        )
    ]["split_group"].tolist()

    for group_id in train_ids:
        split_assignments[group_id] = "train"

    for group_id in val_ids:
        split_assignments[group_id] = "validation"

    for group_id in test_ids:
        split_assignments[group_id] = "test"

    allocation_details.append({
        "pathology": pathology,
        "train_groups": len(train_ids),
        "validation_groups": len(val_ids),
        "test_groups": len(test_ids),
        "validation_images": val_images,
        "test_images": test_images,
        "target_eval_images": target_eval_images
    })

In [60]:
allocation_details = pd.DataFrame(allocation_details)

allocation_details

,pathology,train_groups,validation_groups,test_groups,validation_images,test_images,target_eval_images
0,Astrocytoma,39,9,9,168,168,167.70
1,Ependymoma,17,3,3,156,156,155.55
2,Glioma,35,8,8,221,221,221.40
3,Hemangiopericytoma,7,2,2,89,86,90.45
4,Meningioma,45,10,10,311,311,310.65
5,Neurocytoma,11,2,2,97,85,92.70
6,Normal,10,2,2,160,148,158.70
7,Oligodendroglioma,16,4,4,82,83,82.50
8,Other,39,9,9,230,230,229.95
9,Schwannoma,28,6,6,185,185,185.40


In [61]:
group_df["split"] = group_df["split_group"].map(split_assignments)

group_df["split"].value_counts()

split
train         247
validation     55
test           55
Name: count, dtype: int64

In [62]:
df["split"] = df["split_group"].map(split_assignments)

df["split"].value_counts()

split
train         7928
validation    1699
test          1673
Name: count, dtype: int64

In [63]:
split_summary_optimized = (
    df.groupby("split")
      .agg(
          n_images=("relative_path", "size"),
          n_groups=("split_group", "nunique")
      )
)

split_summary_optimized["pct_images"] = (
    split_summary_optimized["n_images"] / len(df) * 100
)

split_summary_optimized["pct_groups"] = (
    split_summary_optimized["n_groups"]
    / df["split_group"].nunique()
    * 100
)

split_summary_optimized

,n_images,n_groups,pct_images,pct_groups
split,,,,
test,1673,55,14.805310,15.406162
train,7928,247,70.159292,69.187675
validation,1699,55,15.035398,15.406162


In [64]:
pathology_split_optimized = (
    df.groupby(["pathology", "split"])
      .agg(
          n_images=("relative_path", "size"),
          n_groups=("split_group", "nunique")
      )
      .unstack(fill_value=0)
)

pathology_split_optimized

n_images                  n_groups                 
split                  test train validation     test train validation
pathology                                                             
Astrocytoma             168   782        168        9    39          9
Ependymoma              156   725        156        3    17          3
Glioma                  221  1034        221        8    35          8
Hemangiopericytoma       86   428         89        2     7          2
Meningioma              311  1449        311       10    45         10
Neurocytoma              85   436         97        2    11          2
Normal                  148   750        160        2    10          2
Oligodendroglioma        83   385         82        4    16          4
Other                   230  1073        230        9    39          9
Schwannoma              185   866        185        6    28          6

In [65]:
assert df["split"].isna().sum() == 0

assert (
    df.groupby("split_group")["split"]
      .nunique()
      .max()
    == 1
)

assert (
    df.groupby("pixel_hash")["split"]
      .nunique()
      .max()
    == 1
)

assert (
    df.groupby(["pathology", "split"])
      .size()
      .unstack(fill_value=0)
      .gt(0)
      .all()
      .all()
)

print("✓ Every image received a split.")
print("✓ No split_group crosses partitions.")
print("✓ No exact pixel duplicate crosses partitions.")
print("✓ Every pathology is represented in every partition.")

✓ Every image received a split.
✓ No split_group crosses partitions.
✓ No exact pixel duplicate crosses partitions.
✓ Every pathology is represented in every partition.


### 3.3 Final split validation

The optimized allocation closely matches the target 70/15/15 distribution at
both the reconstructed-group and image levels.

Compared with simple pathology-stratified random splitting, the optimized
allocation substantially improves pathology-specific representation in the
validation and test sets while preserving group integrity.

Before freezing the partitions, MRI modality distributions are evaluated to
ensure that the optimization did not introduce substantial T1, T1C+, or T2
imbalances.

In [66]:
modality_split_summary = pd.crosstab(
    df["modality"],
    df["split"],
    margins=True
)

modality_split_summary

split,test,train,validation,All
modality,,,,
T1,593,2397,656,3646
T1C+,632,3422,624,4678
T2,448,2109,419,2976
All,1673,7928,1699,11300


In [67]:
modality_split_pct = (
    pd.crosstab(
        df["modality"],
        df["split"],
        normalize="index"
    )
    * 100
)

modality_split_pct.round(2)

split,test,train,validation
modality,,,
T1,16.26,65.74,17.99
T1C+,13.51,73.15,13.34
T2,15.05,70.87,14.08


## 4. Freeze the classification manifest

The final partitioning strategy was accepted after evaluating pathology,
reconstructed-group, image-count, and MRI modality distributions.

The resulting dataset contains:

- 7,928 training images;
- 1,699 validation images;
- 1,673 test images.

All 357 reconstructed groups are assigned exclusively to one partition, and no
exact pixel duplicate crosses partition boundaries.

MRI modalities remain represented across all partitions. Minor deviations from
the target 70/15/15 proportions were retained rather than further optimizing
the split, in order to avoid unnecessary complexity and preserve the primary
pathology- and group-level constraints.

These partitions are now frozen and will be reused unchanged across subsequent
classification model experiments.

In [68]:
manifest_columns = [
    "relative_path",
    "pathology",
    "modality",
    "split_group",
    "split",
    "width",
    "height",
    "point",
    "location",
    "description",
    "pixel_hash",
]

classification_manifest = (
    df[manifest_columns]
    .copy()
    .sort_values(["split", "pathology", "split_group", "relative_path"])
    .reset_index(drop=True)
)

classification_manifest.head()

,relative_path,pathology,modality,split_group,split,width,height,point,location,description,pixel_hash
0,Astrocytoma T1C+\T1C+ - Diffuse astrocytoma fr...,Astrocytoma,T1C+,9,test,512,512,"{'x': 328, 'y': 150}","[frontal, ventricle]","In the left frontal lobe, there´s an intra-axi...",4426b6ab1d355bb7007e793bea34c592437118bff6d766...
1,Astrocytoma T1C+\T1C+ - Diffuse astrocytoma fr...,Astrocytoma,T1C+,9,test,512,512,"{'x': 324, 'y': 148}","[frontal, ventricle]","In the left frontal lobe, there´s an intra-axi...",bb287c29aa575b46674ccf89141a3a9d0608af47f80bed...
2,Astrocytoma T1C+\T1C+ - Diffuse astrocytoma fr...,Astrocytoma,T1C+,9,test,512,512,"{'x': 328, 'y': 142}","[frontal, ventricle]","In the left frontal lobe, there´s an intra-axi...",22b7ecd552e34cd825f1a378e6f44fdb6ece3f8301a52c...
3,Astrocytoma T1C+\T1C+ - Diffuse astrocytoma fr...,Astrocytoma,T1C+,9,test,512,512,"{'x': 347, 'y': 148}","[frontal, ventricle]","In the left frontal lobe, there´s an intra-axi...",d13368e89e8179f39bd06fd5edbabafb3b9e6a4974e0a9...
4,Astrocytoma T1\T1 - Diffuse astrocytoma fronta...,Astrocytoma,T1,9,test,512,512,"{'x': 318, 'y': 153}","[frontal, ventricle]","In the left frontal lobe, there´s an intra-axi...",442dd55ce6e3f167adcf58c8156ebb5045ec12c6d471bb...


In [69]:
assert len(classification_manifest) == 11300
assert classification_manifest["relative_path"].nunique() == 11300
assert classification_manifest["split_group"].nunique() == 357
assert classification_manifest["split"].isna().sum() == 0

print("✓ Manifest contains all 11,300 images.")
print("✓ Every image is uniquely represented.")
print("✓ All 357 split groups are preserved.")
print("✓ Every image has a final partition.")

✓ Manifest contains all 11,300 images.
✓ Every image is uniquely represented.
✓ All 357 split groups are preserved.
✓ Every image has a final partition.


In [ ]:
MANIFEST_PATH = (
    PROCESSED_DATA_DIR / "classification_manifest.csv"
)

classification_manifest.to_csv(
    MANIFEST_PATH,
    index=False
)


Manifest saved to: /home/swsq2526/brain-tumor-mri/data/processed/classification_manifest.csv


In [71]:
saved_manifest = pd.read_csv(MANIFEST_PATH)

print(saved_manifest.shape)
print(saved_manifest["split"].value_counts())

assert saved_manifest.shape[0] == 11300
assert saved_manifest["relative_path"].nunique() == 11300

print("✓ Saved manifest successfully reloaded.")

(11300, 11)
split
train         7928
validation    1699
test          1673
Name: count, dtype: int64
✓ Saved manifest successfully reloaded.


## 5. Shared image preprocessing strategy

The experimental split is now frozen.

The remaining preparation steps define only transformations that are shared
across model families.

Model-specific preprocessing requirements, such as architecture-specific input
normalization or final input resolution, will be handled within each model
pipeline rather than hard-coded into this shared preparation notebook.

The shared preprocessing stage therefore focuses on:

- validating image readability;
- standardizing image channel format;
- defining a consistent image-loading convention;
- preserving the original MRI content without introducing aggressive
  preprocessing choices that could remove clinically relevant information;
- documenting transformations that must remain identical across model
  comparisons.

In [ ]:

image_modes = Counter()

for path in df["image_path"]:
    with Image.open(path) as img:
        image_modes[img.mode] += 1

image_modes

Counter({'RGB': 11300})

### Image format validation

All 11,300 MRI images are stored as three-channel RGB images.

No channel conversion is therefore required as part of the shared data
preparation pipeline. The original RGB representation will be preserved for
all classification experiments.

Architecture-specific operations such as input resizing and normalization will
be defined within the corresponding model pipelines rather than applied to the
stored dataset.

In [73]:
df["resolved_relative_path"] = df["image_path"].apply(
    lambda p: p.relative_to(PROJECT_ROOT).as_posix()
)

df[
    ["relative_path", "resolved_relative_path"]
].head()

,relative_path,resolved_relative_path
0,Astrocytoma T1\T1 - Anaplastic astrocytoma (pi...,data/raw/Astrocytoma T1/T1 - Anaplastic astroc...
1,Astrocytoma T1\T1 - Anaplastic astrocytoma (pi...,data/raw/Astrocytoma T1/T1 - Anaplastic astroc...
2,Astrocytoma T1\T1 - Anaplastic astrocytoma (pi...,data/raw/Astrocytoma T1/T1 - Anaplastic astroc...
3,Astrocytoma T1\T1 - Anaplastic astrocytoma (pi...,data/raw/Astrocytoma T1/T1 - Anaplastic astroc...
4,Astrocytoma T1\T1 - Anaplastic astrocytoma fro...,data/raw/Astrocytoma T1/T1 - Anaplastic astroc...


In [74]:
manifest_columns = [
    "relative_path",
    "resolved_relative_path",
    "pathology",
    "modality",
    "split_group",
    "split",
    "width",
    "height",
    "point",
    "location",
    "description",
    "pixel_hash",
]

classification_manifest = (
    df[manifest_columns]
    .copy()
    .sort_values(
        ["split", "pathology", "split_group", "relative_path"]
    )
    .reset_index(drop=True)
)

In [ ]:
classification_manifest.to_csv(
    MANIFEST_PATH,
    index=False
)


Manifest saved to: /home/swsq2526/brain-tumor-mri/data/processed/classification_manifest.csv


In [76]:
saved_manifest = pd.read_csv(MANIFEST_PATH)

resolved_paths = saved_manifest["resolved_relative_path"].apply(
    lambda p: PROJECT_ROOT / p
)

print("Manifest shape:", saved_manifest.shape)
print(
    "Resolved files found:",
    resolved_paths.apply(lambda p: p.exists()).sum()
)

Manifest shape: (11300, 12)
Resolved files found: 11300


In [77]:
assert saved_manifest.shape == (11300, 12)
assert saved_manifest["resolved_relative_path"].nunique() == 11300
assert resolved_paths.apply(lambda p: p.exists()).all()

assert (
    saved_manifest.groupby("split_group")["split"]
    .nunique()
    .max()
    == 1
)

assert (
    saved_manifest.groupby("pixel_hash")["split"]
    .nunique()
    .max()
    == 1
)

print("✓ Manifest contains 11,300 images.")
print("✓ All resolved paths are portable and valid.")
print("✓ No split_group crosses partitions.")
print("✓ No exact duplicate crosses partitions.")

✓ Manifest contains 11,300 images.
✓ All resolved paths are portable and valid.
✓ No split_group crosses partitions.
✓ No exact duplicate crosses partitions.


## 6. Final data preparation summary

The brain tumor MRI classification dataset is now prepared for reproducible
model development.

The final experimental dataset contains 11,300 RGB MRI images assigned to
357 reconstructed leakage-aware groups.

A group-size-aware, pathology-stratified allocation produced fixed partitions
of:

- 7,928 training images;
- 1,699 validation images;
- 1,673 test images.

All images belonging to the same reconstructed group remain within a single
partition, and no exact pixel duplicate crosses train, validation, or test
boundaries.

MRI modalities remain represented across all partitions, while model-specific
operations such as resizing, normalization, and data augmentation are
intentionally excluded from the shared preparation pipeline.

The final `classification_manifest.csv` stores both the original metadata path
and a resolved project-relative image path, allowing subsequent model notebooks
to reuse the exact same dataset partitions without repeating path correction,
group reconstruction, or split logic.

This manifest will serve as the fixed experimental reference for all subsequent
classification architectures.